In [ ]:
import matrix_helpers as M


def new_board():
    board = M.make_matrix(7, default=1)
    M.set_item(board, (3, 3), 0)
    return board


def as_tuple(mat):
    return tuple(tuple(row) for row in mat)


def is_pos(pos):
    return all(0 <= i < 7 for i in pos) and any(2 <= i < 5 for i in pos)


def get_pegmoves(board, jpeg):
    x, y = jpeg
    moves = []
    for dx, dy in ((0, 1), (1, 0), (0, -1), (-1, 0)):
        peg, hole = (x+dx, y+dy), (x+2*dx, y+2*dy)
        if is_pos(peg) and M.get_item(board, peg) == 1 and is_pos(hole) and M.get_item(board, hole) == 0:
            moves.append((jpeg, peg, hole))
    return moves


def get_neighbors(board):
    for jpeg, val in M.pos_and_values(board):
        if not is_pos(jpeg) or val == 0:
            continue

        moves = get_pegmoves(board, jpeg)
        for move in moves:
            jpeg, peg, hole = move
            new_board = M.copy(board)
            M.set_item(new_board, jpeg, 0)
            M.set_item(new_board, peg, 0)
            M.set_item(new_board, hole, 1)
            yield move, new_board

In [ ]:
board = new_board()
board[3][0] = 0
board[1][2] = 0
board

In [ ]:
jpeg = (1, 3)
get_pegmoves(board, jpeg)

In [ ]:
jpeg = (2, 3)
get_pegmoves(board, jpeg)

In [ ]:
ns = get_neighbors(board)
ns

In [ ]:
board

In [ ]:
move, board = next(ns)
print(move)
board

In [48]:
def search_df(board, get_neighbors):
    n_pegs = 32
    nodes_to_visit = [(n_pegs, board)]
    go_back = {as_tuple(board): None}

    while nodes_to_visit:
        n_pegs, board = nodes_to_visit.pop()
        if n_pegs == 1 and M.get_item(board, (3, 3)) == 1:
            return board, go_back

        for move, new_board in get_neighbors(board):
            if (key := as_tuple(new_board)) in go_back:
                continue

            go_back[key] = (move, as_tuple(board))
            nodes_to_visit.append((n_pegs - 1, new_board))


def get_path_home(board, go_back):
    node = go_back[as_tuple(board)]
    path = []

    while node:
        jpeg, peg, hole = node[0]
        path.append((jpeg, hole))
        node = go_back[node[1]]

    return path

In [49]:
board, go_back = search_df(new_board(), get_neighbors)
len(go_back)

1168

In [50]:
get_path_home(board, go_back)[::-1]

[((3, 5), (3, 3)),
 ((5, 4), (3, 4)),
 ((4, 6), (4, 4)),
 ((2, 6), (4, 6)),
 ((3, 4), (5, 4)),
 ((6, 4), (4, 4)),
 ((2, 4), (2, 6)),
 ((0, 4), (2, 4)),
 ((4, 3), (4, 5)),
 ((4, 6), (4, 4)),
 ((6, 3), (4, 3)),
 ((4, 3), (4, 5)),
 ((2, 3), (4, 3)),
 ((0, 3), (2, 3)),
 ((2, 3), (2, 5)),
 ((2, 6), (2, 4)),
 ((4, 2), (4, 4)),
 ((4, 5), (4, 3)),
 ((6, 2), (4, 2)),
 ((3, 2), (5, 2)),
 ((2, 1), (2, 3)),
 ((0, 2), (2, 2)),
 ((4, 0), (4, 2)),
 ((4, 3), (4, 1)),
 ((2, 0), (4, 0)),
 ((4, 0), (4, 2)),
 ((5, 2), (3, 2)),
 ((3, 2), (1, 2)),
 ((2, 4), (2, 2)),
 ((1, 2), (3, 2)),
 ((3, 1), (3, 3))]